In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime

# To store data
data = []

# Function to scrape Amazon India
def scrape_amazon(search_query, max_pages=3):
    print(f"Scraping Amazon India for '{search_query}'...")
    
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920x1080")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    base_url = "https://www.amazon.in/s?k=" + search_query.replace(" ", "+")
    
    for page in range(1, max_pages + 1):
        url = f"{base_url}&page={page}"
        driver.get(url)
        time.sleep(3)  # Allow page to load

        products = driver.find_elements(By.XPATH, "//div[@data-component-type='s-search-result']")
        for product in products:
            try:
                title = product.find_element(By.XPATH, ".//span[@class='a-text-normal']").text
                price = product.find_element(By.XPATH, ".//span[@class='a-price-whole']").text
                link = product.find_element(By.XPATH, ".//a[@class='a-link-normal']").get_attribute("href")
                
                data.append({
                    'Title': title,
                    'Price': f"₹{price}",
                    'Link': link,
                    'Source': 'Amazon India'
                })
            except Exception as e:
                continue  # Skip if any field is missing

        print(f"✅ Page {page} scraped ({len(products)} products found)")
    
    driver.quit()

# Save data to CSV
def save_to_csv():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"amazon_data_{timestamp}.csv"
    
    try:
        df = pd.DataFrame(data)
        df.to_csv(filename, index=False)
        print(f"\n✅ Data saved to '{filename}' ✅")
    except Exception as e:
        print(f"\n❌ Failed to save data: {e}")

# Start scraping
def main():
    search_query = "laptop"
    scrape_amazon(search_query)
    save_to_csv()
    print(f"\n✅ Total records collected: {len(data)}")

if __name__ == "__main__":
    main()


Scraping Amazon India for 'laptop'...
✅ Page 1 scraped (16 products found)
✅ Page 2 scraped (0 products found)
✅ Page 3 scraped (0 products found)

✅ Data saved to 'amazon_data_20250319_120813.csv' ✅

✅ Total records collected: 0
